# Sentiment Analysis

### Importing modules

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

### Loading dataframe

In [45]:
df = pd.read_csv('C:/Users/MihajloTesic/Desktop/Python fajlovi/IMDB-Dataset.csv')

### Splitting into train sets and test sets

In [46]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Converting "review" into numerical values using tf-idf

In [47]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## Logistic regression

In [48]:
model1 = LogisticRegression(max_iter=1000)

model1.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [49]:
y_pred1 = model1.predict(X_test_tfidf)

In [50]:
print('Accuracy: ', accuracy_score(y_test, y_pred1))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred1))

Accuracy:  0.8889333333333334

Classification Report:

              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      7411
    positive       0.88      0.90      0.89      7589

    accuracy                           0.89     15000
   macro avg       0.89      0.89      0.89     15000
weighted avg       0.89      0.89      0.89     15000



## L2 (Ridge) Regularization

In [51]:
model2 = LogisticRegression(solver='saga', penalty='l2', C=0.01)

model2.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'saga'
,max_iter,100
,multi_class,'deprecated'


In [52]:
y_pred2 = model2.predict(X_test_tfidf)

In [53]:
print('Accuracy: ', accuracy_score(y_test, y_pred2))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred2))

Accuracy:  0.8294666666666667

Classification Report:

              precision    recall  f1-score   support

    negative       0.85      0.80      0.82      7411
    positive       0.81      0.86      0.84      7589

    accuracy                           0.83     15000
   macro avg       0.83      0.83      0.83     15000
weighted avg       0.83      0.83      0.83     15000



## L1 (Lasso) Regularization

In [54]:
model3 = LogisticRegression(solver='saga', penalty='l1', C=0.01)

model3.fit(X_train_tfidf, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'saga'
,max_iter,100
,multi_class,'deprecated'


In [55]:
y_pred3 = model3.predict(X_test_tfidf)

In [56]:
print('Accuracy: ', accuracy_score(y_test, y_pred3))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred3))

Accuracy:  0.6557333333333333

Classification Report:

              precision    recall  f1-score   support

    negative       0.79      0.41      0.54      7411
    positive       0.61      0.90      0.72      7589

    accuracy                           0.66     15000
   macro avg       0.70      0.65      0.63     15000
weighted avg       0.70      0.66      0.63     15000



## GridSearchCV

In [57]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1,1),(1,2)],
    'tfidf__min_df': [1, 5],
    'tfidf__max_df': [0.8, 0.9],
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__penalty': ['l2']
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)

In [58]:
X_small = X_train[:10000]
y_small = y_train[:10000]

In [59]:
grid.fit(X_small, y_small)

Fitting 3 folds for each of 96 candidates, totalling 288 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'clf__C': [0.01, 0.1, ...], 'clf__penalty': ['l2'], 'tfidf__max_df': [0.8, 0.9], 'tfidf__max_features': [3000, 5000, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


## RandomizedSearchCV

In [60]:
random_search = RandomizedSearchCV(pipeline,param_distributions=param_grid, n_iter=10, cv=3, n_jobs=-1, verbose=2)

In [61]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,estimator,Pipeline(step..._iter=1000))])
,param_distributions,"{'clf__C': [0.01, 0.1, ...], 'clf__penalty': ['l2'], 'tfidf__max_df': [0.8, 0.9], 'tfidf__max_features': [3000, 5000, ...], ...}"
,n_iter,10
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [62]:
def predict_sentiment(text, model=None):
    if model == None:
        model = model1
    text_tfidf = vectorizer.transform([text])
    prediction = model.predict(text_tfidf)
    return prediction[0]

def predict_text(text, model=None):
    if model == None:
        model = grid
    return grid.predict([text])[0]

In [63]:
print(predict_sentiment('This was a good film.'))
print(predict_sentiment('This was a horrible film.'))
print(predict_sentiment('The film was so good, that I had to turn it off.'))
print(predict_sentiment('The film was good, although there were some horrible scenes.'))
print(predict_sentiment('A bad ending, but I loved the film.'))
print(predict_sentiment('I was fascinated when he saved them, but I didn\'t like the film.'))

positive
negative
positive
negative
positive
negative


In [64]:
print(predict_sentiment('This was a good film.', model2))
print(predict_sentiment('This was a horrible film.', model2))
print(predict_sentiment('The film was so good, that I had to turn it off.', model2))
print(predict_sentiment('The film was good, although there were some horrible scenes.', model2))
print(predict_sentiment('A bad ending, but I loved the film.', model2))
print(predict_sentiment('I was fascinated when he saved them, but I didn\'t like the film.', model2))

positive
negative
positive
negative
negative
negative


In [65]:
print(predict_sentiment('This was a good film.', model3))
print(predict_sentiment('This was a horrible film.', model3))
print(predict_sentiment('The film was so good, that I had to turn it off.', model3))
print(predict_sentiment('The film was good, although there were some horrible scenes.', model3))
print(predict_sentiment('A bad ending, but I loved the film.', model3))
print(predict_sentiment('I was fascinated when he saved them, but I didn\'t like the film.', model3))

positive
positive
positive
positive
negative
positive


In [66]:
print(predict_text('This was a good film.'))
print(predict_text('This was a horrible film.'))
print(predict_text('The film was so good, that I had to turn it off.'))
print(predict_text('The film was good, although there were some horrible scenes.'))
print(predict_text('A bad ending, but I loved the film.'))
print(predict_text('I was fascinated when he saved them, but I didn\'t like the film.'))

positive
negative
positive
negative
positive
negative


In [67]:
print(predict_text('This was a good film.', random_search))
print(predict_text('This was a horrible film.', random_search))
print(predict_text('The film was so good, that I had to turn it off.', random_search))
print(predict_text('The film was good, although there were some horrible scenes.', random_search))
print(predict_text('A bad ending, but I loved the film.', random_search))
print(predict_text('I was fascinated when he saved them, but I didn\'t like the film.', random_search))

positive
negative
positive
negative
positive
negative
